# 01a · Swiss roll: barrido de β

**Objetivo.** Elegir el ancho del kernel del DMAP congelado **sin usar la carta verdadera**, y documentar por qué la heurística anterior (`beta_factor=16`) cortocircuitaba capas del enrollado.

β = factor × (1 / mediana de distancias² en las entradas estandarizadas de entrenamiento). Para cada semilla de datos y cada factor se registra:

| Tipo | Cantidad | ¿Se usa para elegir? |
|---|---|---|
| sin etiquetas | masa del kernel en pares cuya distancia geodésica en un grafo kNN (k=10) supera 2× la distancia ambiental | **sí** |
| sin etiquetas | controles espectrales: residuos, separación de Perron, error Nyström, λ mínimo | **sí** (deben pasar en todas las semillas) |
| sin etiquetas | β-flow clásico (dimensión efectiva) | no: su máximo cae *dentro* del régimen de cortocircuito |
| diagnóstico (usa la carta) | masa real entre capas, corr(ψ₁, cos πs/W), posición del modo de altura, oráculo afín, oráculo → g exacta | **no** |

**Regla:** el menor factor (kernel más ancho) cuya peor masa de atajos sobre las semillas es ≤ 10⁻³ y cuyos controles espectrales pasan en todas las semillas.

Etapa 1 (solo espectral, 2 GPU; 6 semillas × 16 factores con m hasta 256): ~10–20 min. Etapa 2 (opcional, redes a varios β): ~25–35 min.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## Etapa 1 · Barrido espectral

In [ ]:
SWEEP_CONFIG = ROOT / 'config_beta_sweep.json'
cfg_sweep = json.loads(SWEEP_CONFIG.read_text())
# Edita aquí si hace falta (p. ej. más semillas de datos o factores). Todo lo demás se deriva.
# cfg_sweep['data_seeds'] = [7, 1001, 1002, 1003, 1004, 1005]
# cfg_sweep['factors'] = [4, 8, 12, 16, 20, 24, 32, 40, 48, 64, 96, 128, 192, 256, 384, 512]
ACTIVE_SWEEP = ROOT / 'active_beta_sweep.json'
ACTIVE_SWEEP.write_text(json.dumps(cfg_sweep, indent=2))
SWEEP_OUT = Path(cfg_sweep['out'])
print(json.dumps(cfg_sweep, indent=2))

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'beta_sweep.py'), str(ACTIVE_SWEEP), '--launch'],
                   capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode:
    print(r.stderr[-6000:])
    for log in sorted(SWEEP_OUT.glob('sweep_worker*.log')):
        print('---', log.name); print(log.read_text()[-3000:])
    raise RuntimeError('Falló el barrido de beta.')
print(f'Barrido: {(time.time()-t0)/60:.1f} min')

In [ ]:
import pandas as pd
from IPython.display import display, Image
selection = json.loads((SWEEP_OUT / 'beta_selection.json').read_text())
print(json.dumps(selection, indent=2))
df_sweep = pd.read_csv(SWEEP_OUT / 'beta_sweep.csv')
cols = ['kernel_length', 'graph_shortcut_mass', 'true_shortcut_mass', 'psi1_corr_cos_s',
        'height_mode_position', 'perron_gap', 'oracle_val_m2', 'oracle_val_m8', 'oracle_val_m32',
        'oracle_exact_x_m8', 'oracle_exact_x_m256']
cols = [c for c in cols if c in df_sweep]
summary = df_sweep.groupby('factor').agg(ok=('ok', 'mean'), **{c: (c, 'mean') for c in cols})
display(summary.style.format('{:.3g}'))
display(Image(filename=str(SWEEP_OUT / 'beta_sweep.png')))

## Etapa 2 · Redes neuronales a varios β (opcional)

Mismo protocolo en todos los β: modos 2–32, `ae`, `vae`, `direct`, tres réplicas con datos y partición independientes. Sirve para mostrar el efecto del cortocircuito sobre la reconstrucción; **no** se usa para elegir β.

In [ ]:
sel = selection['selected_factor']
FACTORS = sorted({16, 32, sel, 2*sel})       # 16 = heurística anterior
base = json.loads((ROOT / 'config_swiss.json').read_text())
base.update(modes=[2, 4, 8, 16, 32], models=['ae', 'vae', 'direct'], m_protocols=['scratch'],
            seeds=[0, 1, 2], resample_data=True)
runs = {}
for f in FACTORS:
    cfg = dict(base, beta_factor=f, out=f'/kaggle/working/results_swiss_beta{f}')
    path = ROOT / f'active_swiss_beta{f}.json'
    path.write_text(json.dumps(cfg, indent=2))
    if not (Path(cfg['out']) / 'metrics.csv').exists():
        t0 = time.time()
        subprocess.run([sys.executable, str(ROOT / 'launch.py'), str(path)], check=True)
        print(f'factor {f}: {(time.time()-t0)/60:.1f} min')
    runs[f] = Path(cfg['out'])

In [ ]:
import matplotlib.pyplot as plt
frames = []
for f, folder in runs.items():
    d = pd.read_csv(folder / 'metrics.csv'); d['factor'] = f; frames.append(d)
allruns = pd.concat(frames)
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
for ax, model in zip(axes, ['direct', 'ae', 'vae']):
    for f in FACTORS:
        g = allruns[(allruns.model == model) & (allruns.factor == f)].groupby('m').mse_x.agg(['mean', 'std'])
        ax.errorbar(g.index, g['mean'], yerr=g['std'], marker='o', ms=3, label=f'factor {f}')
    ax.set(xscale='log', yscale='log', xlabel='m', title=f'{model}: test MSE$_x$')
    ax.legend(fontsize=7)
for f in FACTORS:
    g = allruns[(allruns.model == 'oracle') & (allruns.factor == f)].groupby('m').chart_mse_z.mean()
    axes[3].plot(g.index, g.values, 'o-', ms=3, label=f'factor {f}')
axes[3].set(xscale='log', yscale='log', xlabel='m', title='oracle chart MSE$_z$ (diagnostic)')
axes[3].legend(fontsize=7)
fig.tight_layout(); fig.savefig('/kaggle/working/beta_neural_comparison.png', dpi=150); plt.show()
display(allruns[allruns.model.isin(['ae', 'vae', 'direct'])].pivot_table(
    index=['model', 'm'], columns='factor', values='mse_x', aggfunc='mean').style.format('{:.3g}'))

## Qué guardar

`/kaggle/working/beta_sweep_swiss/` (CSV, `beta_selection.json`, figura PNG/PDF, JSON por semilla) y, si corriste la etapa 2, `results_swiss_beta*/` y `beta_neural_comparison.png`. Descarga la salida de la sesión.